In [ ]:
# Cell 1: Install required libraries
!pip install torch torchaudio librosa numpy matplotlib lion-pytorch datasets transformers

# Cell 2: Imports
import torch
import torch.nn as nn
import torch.optim as optim
import torchaudio
import librosa
import numpy as np
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset, Audio
from torch import amp
from lion_pytorch import Lion
import time
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# Load Hindi TTS dataset from HF
print("Loading dataset...")
dataset = load_dataset("SPRINGLab/IndicTTS-Hindi", split="train")
dataset = dataset.cast_column("audio", Audio(sampling_rate=22050))

# Limit dataset size for faster experimentation (remove this line for full training)
dataset = dataset.select(range(min(5000, len(dataset))))  # Use first 5000 samples

print(f"Dataset loaded with {len(dataset)} samples")
print("Example:", dataset[0]["text"])

# Cell 3: Improved Dataset class
class TTSDataset(Dataset):
    def __init__(self, dataset, n_mels=80, sample_rate=22050, max_mel_length=1000):
        self.dataset = dataset
        self.sample_rate = sample_rate
        self.n_mels = n_mels
        self.max_mel_length = max_mel_length
        
        # Build vocabulary with special tokens
        print("Building vocabulary...")
        all_chars = set()
        self.transcripts = []
        
        for item in tqdm(dataset, desc="Processing texts"):
            text = item["text"].strip()
            if text:  # Only add non-empty texts
                self.transcripts.append(text)
                all_chars.update(text)
        
        # Create character mapping with special tokens
        self.char2id = {"<PAD>": 0, "<UNK>": 1}
        for i, char in enumerate(sorted(all_chars), 2):
            self.char2id[char] = i
        
        self.id2char = {v: k for k, v in self.char2id.items()}
        self.vocab_size = len(self.char2id)
        
        print(f"Vocabulary size: {self.vocab_size}")
        print(f"Sample characters: {list(self.char2id.keys())[:20]}")
        
        # Filter dataset to remove items with empty transcripts or very short audio
        valid_indices = []
        for idx in tqdm(range(len(dataset)), desc="Filtering dataset"):
            if (dataset[idx]["text"].strip() and 
                len(dataset[idx]["audio"]["array"]) > sample_rate * 0.5):  # At least 0.5s audio
                valid_indices.append(idx)
        
        self.valid_indices = valid_indices
        print(f"Valid samples after filtering: {len(self.valid_indices)}")
    
    def __len__(self):
        return len(self.valid_indices)
    
    def __getitem__(self, idx):
        actual_idx = self.valid_indices[idx]
        item = self.dataset[actual_idx]
        
        try:
            # Process audio
            wav = item["audio"]["array"]
            
            # Normalize audio
            wav = wav / (np.abs(wav).max() + 1e-6)
            
            # Compute mel spectrogram with better parameters
            mel = librosa.feature.melspectrogram(
                y=wav, 
                sr=self.sample_rate, 
                n_mels=self.n_mels,
                n_fft=1024,
                hop_length=256,
                win_length=1024,
                fmax=8000  # Important for speech
            )
            
            # Convert to log scale and normalize
            mel_db = librosa.power_to_db(mel, ref=np.max)
            mel_db = (mel_db + 100) / 100  # Normalize to roughly [0, 1]
            mel_db = torch.tensor(mel_db, dtype=torch.float).T  # time x n_mels
            
            # Truncate or pad mel spectrogram
            if mel_db.shape[0] > self.max_mel_length:
                mel_db = mel_db[:self.max_mel_length]
            
            # Process text
            text = item["text"].strip()
            text_ids = torch.tensor([
                self.char2id.get(c, self.char2id["<UNK>"]) for c in text
            ], dtype=torch.long)
            
            return text_ids, mel_db
            
        except Exception as e:
            print(f"Error processing item {actual_idx}: {e}")
            # Return a dummy sample
            return torch.tensor([1], dtype=torch.long), torch.zeros(10, self.n_mels)

# Cell 4: Improved TTS Model with Attention
class ImprovedTTS(nn.Module):
    def __init__(self, vocab_size, mel_channels=80, hidden_dim=512):
        super(ImprovedTTS, self).__init__()
        self.hidden_dim = hidden_dim
        self.mel_channels = mel_channels
        
        # Text encoder
        self.embedding = nn.Embedding(vocab_size, 256)
        self.text_encoder = nn.LSTM(256, hidden_dim//2, batch_first=True, bidirectional=True)
        
        # Attention mechanism (simplified)
        self.attention = nn.MultiheadAttention(hidden_dim, num_heads=8, batch_first=True)
        
        # Decoder
        self.decoder_lstm = nn.LSTM(hidden_dim, hidden_dim, batch_first=True, num_layers=2)
        
        # Output projection
        self.mel_projection = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, mel_channels)
        )
        
        # Postnet for refining output
        self.postnet = nn.Sequential(
            nn.Conv1d(mel_channels, 256, kernel_size=5, padding=2),
            nn.BatchNorm1d(256),
            nn.Tanh(),
            nn.Dropout(0.1),
            nn.Conv1d(256, mel_channels, kernel_size=5, padding=2)
        )
        
    def forward(self, text_ids, max_length=None):
        batch_size = text_ids.size(0)
        
        # Text encoding
        embedded = self.embedding(text_ids)  # [B, T_text, 256]
        text_encoded, _ = self.text_encoder(embedded)  # [B, T_text, hidden_dim]
        
        # If max_length not specified, use text length * 10 as approximation
        if max_length is None:
            max_length = min(text_ids.size(1) * 10, 1000)
        
        # Initialize decoder state
        h_0 = torch.zeros(2, batch_size, self.hidden_dim, device=text_ids.device)
        c_0 = torch.zeros(2, batch_size, self.hidden_dim, device=text_ids.device)
        decoder_state = (h_0, c_0)
        
        # Decoder with attention
        decoder_input = torch.zeros(batch_size, 1, self.hidden_dim, device=text_ids.device)
        mel_outputs = []
        
        for t in range(max_length):
            # Decoder step
            decoder_output, decoder_state = self.decoder_lstm(decoder_input, decoder_state)
            
            # Attention
            attended_context, _ = self.attention(
                decoder_output, text_encoded, text_encoded
            )
            
            # Combine decoder output with attended context
            combined = decoder_output + attended_context
            
            # Generate mel frame
            mel_frame = self.mel_projection(combined)
            mel_outputs.append(mel_frame)
            
            # Use current output as next input (teacher forcing during training)
            decoder_input = combined
        
        mel_output = torch.cat(mel_outputs, dim=1)  # [B, T_mel, mel_channels]
        
        # Apply postnet
        mel_postnet = mel_output.transpose(1, 2)  # [B, mel_channels, T_mel]
        mel_residual = self.postnet(mel_postnet)
        mel_postnet = mel_output.transpose(1, 2) + mel_residual
        mel_postnet = mel_postnet.transpose(1, 2)  # [B, T_mel, mel_channels]
        
        return mel_output, mel_postnet

# Cell 5: Optimized training setup
def collate_fn(batch):
    """Improved collate function"""
    # Filter out None samples
    batch = [item for item in batch if item[0].numel() > 0 and item[1].numel() > 0]
    
    if len(batch) == 0:
        # Return dummy batch if all samples are invalid
        return {
            'text_ids': torch.zeros(1, 1, dtype=torch.long),
            'mels': torch.zeros(1, 10, 80),
            'text_lengths': torch.tensor([1]),
            'mel_lengths': torch.tensor([10])
        }
    
    text_ids_list, mels_list = zip(*batch)
    
    # Pad sequences
    text_ids = nn.utils.rnn.pad_sequence(text_ids_list, batch_first=True, padding_value=0)
    mels = nn.utils.rnn.pad_sequence(mels_list, batch_first=True, padding_value=0)
    
    # Get lengths
    text_lengths = torch.tensor([len(seq) for seq in text_ids_list])
    mel_lengths = torch.tensor([len(seq) for seq in mels_list])
    
    return {
        'text_ids': text_ids,
        'mels': mels,
        'text_lengths': text_lengths,
        'mel_lengths': mel_lengths
    }

# Create dataset and dataloader
print("Creating dataset...")
tts_dataset = TTSDataset(dataset)

dataloader = DataLoader(
    tts_dataset,
    batch_size=16,  # Smaller batch size for complex model
    shuffle=True,
    collate_fn=collate_fn,
    pin_memory=True,
    num_workers=4,  # Reduced for stability
    persistent_workers=True,
    prefetch_factor=2,
    drop_last=True
)

print(f"DataLoader created with {len(dataloader)} batches")

# Initialize model
vocab_size = tts_dataset.vocab_size
model = ImprovedTTS(vocab_size).to(DEVICE)

print(f"Model created with {sum(p.numel() for p in model.parameters()):,} parameters")

# Optimized training setup
optimizer = Lion(model.parameters(), lr=1e-4, weight_decay=1e-2)
criterion = nn.L1Loss()  # L1 loss often works better for mel spectrograms

# Mixed precision scaler
scaler = amp.GradScaler()

# Learning rate scheduler
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, 
    max_lr=1e-4, 
    steps_per_epoch=len(dataloader), 
    epochs=20,  # Reduced for faster experimentation
    pct_start=0.1
)

# Cell 6: Training loop
epochs = 20
accumulation_steps = 2

print("Starting training...")
print(f"Training setup:")
print(f"  - Dataset size: {len(tts_dataset)}")
print(f"  - Batch size: {dataloader.batch_size}")
print(f"  - Effective batch size: {dataloader.batch_size * accumulation_steps}")
print(f"  - Steps per epoch: {len(dataloader)}")

for epoch in range(1, epochs + 1):
    model.train()
    running_loss = 0.0
    running_postnet_loss = 0.0
    epoch_start_time = time.time()
    
    pbar = tqdm(enumerate(dataloader), total=len(dataloader), 
                desc=f"Epoch {epoch}/{epochs}")
    
    for batch_idx, batch in pbar:
        try:
            # Move data to device
            text_ids = batch['text_ids'].to(DEVICE, non_blocking=True)
            mels = batch['mels'].to(DEVICE, non_blocking=True)
            mel_lengths = batch['mel_lengths']
            
            # Mixed precision forward pass
            with amp.autocast(device_type='cuda'):
                # Forward pass
                mel_pred, mel_postnet = model(text_ids, max_length=mels.size(1))
                
                # Calculate losses with proper masking
                batch_losses = []
                postnet_losses = []
                
                for i, target_len in enumerate(mel_lengths):
                    actual_len = min(mel_pred.size(1), target_len.item())
                    if actual_len > 1:
                        # Main loss
                        loss_main = criterion(
                            mel_pred[i, :actual_len, :], 
                            mels[i, :actual_len, :]
                        )
                        batch_losses.append(loss_main)
                        
                        # Postnet loss
                        loss_postnet = criterion(
                            mel_postnet[i, :actual_len, :], 
                            mels[i, :actual_len, :]
                        )
                        postnet_losses.append(loss_postnet)
                
                if batch_losses:
                    loss_main = torch.stack(batch_losses).mean()
                    loss_postnet = torch.stack(postnet_losses).mean()
                    loss = loss_main + loss_postnet
                    loss = loss / accumulation_steps
                else:
                    continue
            
            # Backward pass
            scaler.scale(loss).backward()
            
            # Update weights
            if (batch_idx + 1) % accumulation_steps == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad()
            
            running_loss += loss_main.item() if batch_losses else 0
            running_postnet_loss += loss_postnet.item() if postnet_losses else 0
            
            # Update progress bar
            pbar.set_postfix({
                'loss': f'{loss_main.item():.4f}' if batch_losses else 'N/A',
                'postnet': f'{loss_postnet.item():.4f}' if postnet_losses else 'N/A',
                'lr': f'{scheduler.get_last_lr()[0]:.2e}'
            })
            
        except Exception as e:
            print(f"Error in batch {batch_idx}: {e}")
            continue
    
    # Epoch summary
    avg_loss = running_loss / len(dataloader)
    avg_postnet_loss = running_postnet_loss / len(dataloader)
    epoch_time = time.time() - epoch_start_time
    
    print(f"Epoch {epoch}/{epochs} completed in {epoch_time:.1f}s")
    print(f"  Main Loss: {avg_loss:.4f}")
    print(f"  Postnet Loss: {avg_postnet_loss:.4f}")
    print(f"  Learning Rate: {scheduler.get_last_lr()[0]:.2e}")
    
    # Save checkpoint every 5 epochs
    if epoch % 5 == 0:
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'char2id': tts_dataset.char2id,
            'vocab_size': vocab_size,
            'loss': avg_loss
        }
        torch.save(checkpoint, f'improved_tts_checkpoint_epoch_{epoch}.pth')
        print(f"  Checkpoint saved: improved_tts_checkpoint_epoch_{epoch}.pth")
    
    print("-" * 60)

print("Training completed!")

# Cell 7: Improved inference
def synthesize_improved(model, text, char2id, max_length=500):
    """Improved synthesis function"""
    model.eval()
    with torch.no_grad():
        # Convert text to IDs
        text_ids = torch.tensor([
            char2id.get(c, char2id.get("<UNK>", 1)) for c in text
        ], dtype=torch.long).unsqueeze(0).to(DEVICE)
        
        # Generate mel spectrogram
        mel_pred, mel_postnet = model(text_ids, max_length=max_length)
        
        # Use postnet output for better quality
        mel_output = mel_postnet[0].cpu().numpy()  # [T, n_mels]
        mel_output = mel_output.T  # [n_mels, T]
        
        # Denormalize
        mel_output = mel_output * 100 - 100
        
        # Convert mel to waveform using Griffin-Lim
        mel_spec = librosa.db_to_power(mel_output)
        wav = librosa.feature.inverse.mel_to_audio(
            mel_spec, 
            sr=22050, 
            n_iter=64,  # More iterations for better quality
            hop_length=256,
            n_fft=1024
        )
        
    return wav

# Test synthesis
if len(tts_dataset) > 0:
    test_text = "नमस्ते, मैं एक टीटीएस मॉडल हूं।"  # "Hello, I am a TTS model."
    print(f"Synthesizing: {test_text}")
    
    try:
        wav_output = synthesize_improved(model, test_text, tts_dataset.char2id)
        
        # Save output
        torchaudio.save("improved_tts_output.wav", 
                       torch.tensor(wav_output).unsqueeze(0).float(), 22050)
        print("✓ Audio saved as 'improved_tts_output.wav'")
        
        # Also try with a simpler text
        simple_text = "हैलो"  # "Hello"
        wav_simple = synthesize_improved(model, simple_text, tts_dataset.char2id)
        torchaudio.save("simple_tts_output.wav", 
                       torch.tensor(wav_simple).unsqueeze(0).float(), 22050)
        print("✓ Simple audio saved as 'simple_tts_output.wav'")
        
    except Exception as e:
        print(f"Synthesis error: {e}")

print("All done! 🎉")